# 06 - GRPO on GSM8K from base Qwen2.5-1.5B-Instruct (Option A)

## 1. Installations

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq math-verify
print("is_t4:", is_t4, "| vllm:", _vllm)

## 2. Environment check

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_TF"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import importlib.metadata as md
for pkg in ["torch", "vllm", "transformers", "trl", "peft", "unsloth", "datasets", "math-verify"]:
    try: print(f"{pkg:13s} {md.version(pkg)}")
    except md.PackageNotFoundError: print(f"{pkg:13s} NOT INSTALLED")

import torch
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Shared extractor + eval helpers

In [ ]:
%%writefile rf_extract.py
import re

BOXED_SUFFIX = "\nPlease reason step by step, and put your final answer within \\boxed{}."


def user_prompt(question, style="boxed"):
    if style == "boxed":
        return question + BOXED_SUFFIX
    if style == "plain":
        return question
    raise ValueError(style)


def _boxed_spans(text):
    key = r"\boxed{"
    spans, i = [], 0
    while True:
        j = text.find(key, i)
        if j == -1:
            break
        k = j + len(key)
        start, depth = k, 1
        while k < len(text) and depth > 0:
            if text[k] == "{":
                depth += 1
            elif text[k] == "}":
                depth -= 1
            k += 1
        spans.append(text[start:k - 1])
        i = k
    return spans


def _unwrap_text(s):
    return re.sub(r"\\(?:text|textbf|mbox|mathrm)\{([^{}]*)\}", r"\1", s)


def norm(x):
    if x is None:
        return None
    s = str(x).strip()
    s = _unwrap_text(s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\;", "").replace(" ", "")
    s = s.replace("\\$", "").replace("$", "")
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = s.replace("\\%", "").replace("%", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "")
    s = s.rstrip(".").strip()
    if s.startswith("{") and s.endswith("}"):
        s = s[1:-1]
    return s


def extract_boxed(text):
    boxes = _boxed_spans(text)
    return norm(boxes[-1]) if boxes else None


def extract_answer(text):
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return norm(nums[-1]) if nums else None


def _to_float(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def _math_verify(pred, gold):
    try:
        from math_verify import parse, verify
    except ImportError:
        return False
    try:
        return bool(verify(parse("\\boxed{" + gold + "}"), parse("\\boxed{" + pred + "}")))
    except Exception:
        return False


def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    p, g = norm(pred), norm(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    gf = _to_float(g)
    if gf is not None:
        m = re.fullmatch(r"(-?\d+(?:\.\d+)?)[a-zA-Z]*", p)
        pf = _to_float(m.group(1)) if m else None
        return pf is not None and abs(pf - gf) < 1e-6
    return _math_verify(p, g)


def gsm8k_gold(answer_field):
    return norm(answer_field.split("####")[-1])


def has_boxed(text):
    return bool(_boxed_spans(text))

In [ ]:
%%writefile rf_eval.py
import os
import pickle

from rf_extract import answers_match, extract_answer, gsm8k_gold, has_boxed, user_prompt


def load_eval_set(name, n=None):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        items = [{"question": r["question"], "gold": gsm8k_gold(r["answer"])} for r in ds]
    elif name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [{"question": r["problem"], "gold": r["answer"]} for r in ds]
    else:
        raise ValueError(name)
    return items[:n] if n else items


def chat_prompts(tokenizer, items, style):
    return [tokenizer.apply_chat_template([{"role": "user", "content": user_prompt(it["question"], style)}],
                                          tokenize=False, add_generation_prompt=True)
            for it in items]


def rows_from_outputs(items, outputs):
    rows = []
    for it, out in zip(items, outputs):
        c = out.outputs[0]
        rows.append({"question": it["question"], "gold": it["gold"], "resp": c.text,
                     "n_new_tokens": len(c.token_ids),
                     "truncated": int(c.finish_reason == "length"),
                     "has_box": has_boxed(c.text)})
    return rows


def quick_acc(rows):
    return sum(answers_match(extract_answer(r["resp"]), r["gold"]) for r in rows) / max(len(rows), 1)


def load_result(path, config):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {"config": config}


def save_result(path, result):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(result, f)
    os.replace(tmp, path)

## 4. Config

In [ ]:
SMOKE_TEST = True

RUN_NAME          = "grpo_a"
MODEL_ID          = "unsloth/Qwen2.5-1.5B-Instruct"
LOAD_4BIT         = False
LORA_R, LORA_ALPHA = 32, 64

NUM_GENERATIONS   = 8
PROMPTS_PER_STEP  = 4
MAX_PROMPT_LEN    = 320
MAX_COMPLETION    = 1024
LR                = 1e-5
BETA              = 0.0
NUM_ITERATIONS    = 1
EPSILON, EPSILON_HIGH = 0.2, 0.28
FORMAT_WEIGHT     = 0.1
MAX_STEPS         = 600
SAVE_STEPS        = 25
TIME_BUDGET_HOURS = 10.0

EVAL_SETS         = ["gsm8k", "math500"]
EVAL_N            = None

if SMOKE_TEST:
    MAX_STEPS, SAVE_STEPS, EVAL_N, TIME_BUDGET_HOURS = 5, 5, 50, 1.0

OUT_DIR   = f"/kaggle/working/{RUN_NAME}"
FINAL_DIR = f"/kaggle/working/{RUN_NAME}-final"
print(f"SMOKE_TEST={SMOKE_TEST}  MAX_STEPS={MAX_STEPS}  completions/step={NUM_GENERATIONS*PROMPTS_PER_STEP}")

## 5. Weights & Biases

In [ ]:
REPORT_TO = "none"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_PROJECT"] = "reasonforge"
    REPORT_TO = "wandb"
except Exception as e:
    print("W&B disabled (no WANDB_API_KEY secret):", type(e).__name__)
print("report_to:", REPORT_TO)

## 6. Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_PROMPT_LEN + MAX_COMPLETION + 64,
    load_in_4bit=LOAD_4BIT,
    fast_inference=True,
    max_lora_rank=LORA_R,
    gpu_memory_utilization=0.85,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 7. Data + rewards

In [ ]:
from datasets import load_dataset
from rf_extract import user_prompt, gsm8k_gold, extract_boxed, answers_match, has_boxed

train = load_dataset("openai/gsm8k", "main", split="train")
train = train.map(lambda r: {"prompt": [{"role": "user", "content": user_prompt(r["question"], "boxed")}],
                             "gold": gsm8k_gold(r["answer"])},
                  remove_columns=train.column_names)
print(train)
print(train[0]["prompt"][0]["content"], "\n-> gold:", train[0]["gold"])

_calls = {"n": 0}

def correctness_reward(completions, gold, **kwargs):
    texts = [c[0]["content"] for c in completions]
    rewards = [1.0 if answers_match(extract_boxed(t), g) else 0.0 for t, g in zip(texts, gold)]
    _calls["n"] += 1
    if _calls["n"] % 25 == 1:
        print(f"\n--- sample (call {_calls['n']}) gold={gold[0]} pred={extract_boxed(texts[0])} r={rewards[0]}\n"
              f"{texts[0][-600:]}\n---")
    return rewards

def format_reward(completions, **kwargs):
    return [FORMAT_WEIGHT if has_boxed(c[0]["content"]) else 0.0 for c in completions]

## 8. Trainer

In [ ]:
import glob, json, re, time
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

def latest_checkpoint():
    cands = glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
    cands += glob.glob(f"{OUT_DIR}/checkpoint-*/trainer_state.json")
    cands = [os.path.dirname(c) for c in cands if RUN_NAME in c]
    return max(cands, key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)), default=None)

RESUME = None if SMOKE_TEST else latest_checkpoint()
print("resume from:", RESUME)

class TimeBudgetAndLogs(TrainerCallback):
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
        self.t_last = None
    def on_step_end(self, args, state, control, **kw):
        now = time.time()
        if self.t_last is not None and state.global_step % 5 == 0:
            print(f"step {state.global_step}: {now - self.t_last:.0f}s/step")
        self.t_last = now
        if now > self.deadline:
            print(f"time budget reached at step {state.global_step} -> saving and stopping")
            control.should_save = True
            control.should_training_stop = True
    def on_save(self, args, state, control, **kw):
        with open(f"{args.output_dir}/log_history.json", "w") as f:
            json.dump(state.log_history, f)

args = GRPOConfig(
    output_dir=OUT_DIR,
    run_name=RUN_NAME,
    report_to=REPORT_TO,
    seed=42,
    learning_rate=LR,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=10,
    optim="adamw_8bit",
    adam_beta1=0.9, adam_beta2=0.99,
    weight_decay=0.0,
    max_grad_norm=0.2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=PROMPTS_PER_STEP,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION,
    temperature=1.0,
    use_vllm=True,
    mask_truncated_completions=True,
    loss_type="dapo",
    beta=BETA,
    num_iterations=NUM_ITERATIONS,
    epsilon=EPSILON,
    epsilon_high=EPSILON_HIGH,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward, format_reward],
    args=args,
    train_dataset=train,
    callbacks=[TimeBudgetAndLogs(TIME_BUDGET_HOURS)],
)

## 9. Train

In [ ]:
t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME)
print(f"training wall time: {(time.time()-t0)/3600:.2f} h")

## 10. Save adapter + logs

In [ ]:
import shutil
model.save_lora(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
with open(f"{FINAL_DIR}/log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f)

shutil.make_archive(f"/kaggle/working/{RUN_NAME}_adapter", "zip", FINAL_DIR)
os.makedirs("/kaggle/working/logs", exist_ok=True)
shutil.copy(f"{FINAL_DIR}/log_history.json", f"/kaggle/working/logs/{RUN_NAME}_log_history.json")
shutil.make_archive(f"/kaggle/working/{RUN_NAME}_logs", "zip", "/kaggle/working/logs")
print(os.listdir(FINAL_DIR))

## 11. Eval: base vs GRPO

In [ ]:
from vllm import SamplingParams
from rf_eval import load_eval_set, chat_prompts, rows_from_outputs, quick_acc, load_result, save_result

RES_DIR = "/kaggle/working/results"
os.makedirs(RES_DIR, exist_ok=True)
sp = SamplingParams(temperature=0.0, max_tokens=MAX_COMPLETION, repetition_penalty=1.0)
lora = model.load_lora(FINAL_DIR)

for ds_name in EVAL_SETS:
    items = load_eval_set(ds_name, EVAL_N)
    prompts = chat_prompts(tokenizer, items, "boxed")
    path = f"{RES_DIR}/{ds_name}_boxed_{RUN_NAME}.pkl"
    result = load_result(path, {"dataset": ds_name, "style": "boxed", "n": len(items),
                                "max_new": MAX_COMPLETION, "engine": "unsloth-vllm", "decoding": "greedy"})
    for name, lr in [("base", None), (RUN_NAME, lora)]:
        t1 = time.time()
        outs = model.fast_generate(prompts, sampling_params=sp, lora_request=lr)
        result[name] = rows_from_outputs(items, outs)
        save_result(path, result)
        rows = result[name]
        print(f"{ds_name:8s} {name:8s} acc {quick_acc(rows)*100:5.1f}%  "
              f"trunc {sum(r['truncated'] for r in rows)/len(rows)*100:4.1f}%  "
              f"boxed {sum(r['has_box'] for r in rows)/len(rows)*100:5.1f}%  ({(time.time()-t1)/60:.1f} min)")

shutil.make_archive("/kaggle/working/results", "zip", "/kaggle/working", "results")
print("\ndownload: results.zip, grpo_a_adapter.zip, grpo_a_logs.zip")